# <center>  **<span style="font-size:80px;">Exploración de los Datos</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys, Paths
Paths.init_project()

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor
from src.water2fraud.features.amaem_processor import AMAEMProcessor
preprocessor = WaterPreprocessor()

In [ ]:
seed = 80

images_path = Path("./AMAEM")
images_path.mkdir(parents=True, exist_ok=True)

global_path = images_path / "global"
global_path.mkdir(parents=True, exist_ok=True)
barrios_path = images_path / "barrios"
barrios_path.mkdir(parents=True, exist_ok=True)

# Carga de Datos

In [ ]:
df = pd.read_csv(Paths.RAW_CSV_AMAEM)

# Preprocesamiento

## Renombrado

In [ ]:
df = AMAEMProcessor._rename_df(df)

## Exploración Estructura General

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.head()

In [ ]:
barrios_csv = df[DatasetKeys.BARRIO].unique()
len(barrios_csv)

In [ ]:
barrios_csv

In [ ]:
df.isnull().sum()

## Manejo de Valores Faltantes

In [ ]:
df[df[DatasetKeys.CONSUMO].isna()].sort_values(by=DatasetKeys.BARRIO)

A la hora de realizar el procesamiento sobre nuestros datos deberemos tener en cuenta que los siguientes barrios no cuentan con información sobre el contrato con uso Comercial:

- 35-VIRGEN DEL CARMEN
- TABARCA
- REBOLLEDO

In [ ]:
df = AMAEMProcessor._process_NaN(df)

## Conversión de datos

In [ ]:
df = AMAEMProcessor._convert_dtype(df)
df.dtypes

In [ ]:
df

# Análisis

## DataFrame por Barrios

In [ ]:
cols_to_agg = {
    DatasetKeys.CONSUMO:        "mean",
    DatasetKeys.NUM_CONTRATOS:  "mean",
    DatasetKeys.CONSUMO_RATIO: "mean"
}

df_barrios = df.groupby([DatasetKeys.BARRIO, DatasetKeys.USO]).agg(cols_to_agg).reset_index()

## Datos

In [ ]:
feature, labels_col = DatasetKeys.CONSUMO_RATIO, DatasetKeys.USO

global_labels = df[labels_col].unique()
global_palette = dict(zip(global_labels, sns.color_palette("viridis", len(global_labels))))

barrios_labels = df_barrios[labels_col].unique()
barrios_palette = dict(zip(barrios_labels, sns.color_palette("viridis", len(barrios_labels))))

In [ ]:
def feature_analysis(df, feature, output_path):
    df = df.copy()
    
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(4, 2, figsize=(16, 24))

    for i, ax_row in enumerate(axes):
        is_glb = (i == 3)
        subset = df if is_glb else df[df[labels_col] == global_labels[i]]
        title, color = ("VISTA GLOBAL", None) if is_glb else (global_labels[i], global_palette[global_labels[i]])
        
        # --- CÁLCULO DE LÍMITES CON RECORTE ---
        q1, q3 = subset[feature].quantile(0.25), subset[feature].quantile(0.75)
        iqr = q3 - q1
        
        # Aplicamos max(0, ...) para que el límite inferior nunca sea negativo
        l_inf = max(0, q1 - 1.5 * iqr)
        l_sup = q3 + 1.5 * iqr
        
        # Params comunes
        strip_kwargs = {"hue": labels_col, "palette": global_palette} if is_glb else {"color": color}
        hist_kwargs = {"hue": labels_col, "palette": global_palette, "common_norm": False} if is_glb else {"color": color}

        # COLUMNA 0: Boxplot + Strips
        sns.boxplot(data=subset, x=feature, ax=ax_row[0], color=color or "#f5f5f5", boxprops={"alpha": 0.3}, fliersize=0)
        sns.stripplot(data=subset, x=feature, ax=ax_row[0], alpha=0.3, jitter=0.3, s=5, **strip_kwargs)
        
        # Outliers
        outliers = subset[subset[feature] > l_sup]
        ax_row[0].scatter(outliers[feature], np.zeros(len(outliers)), edgecolor="black", facecolor="none", linewidth=0.6, s=30, zorder=5)
        if is_glb: ax_row[0].get_legend().remove()

        # COLUMNA 1: Histograma
        sns.histplot(data=subset, x=feature, kde=True, ax=ax_row[1], alpha=0.4, bins=50, **hist_kwargs)
        
        # Líneas de umbral
        ax_row[1].axvline(l_sup, color="red", linestyle="--", alpha=0.6, label=f'L.Sup: {l_sup:.1f}')
        if l_inf > 0:
            ax_row[1].axvline(l_inf, color="red", linestyle="--", alpha=0.6, label=f'L.Inf: {l_inf:.1f}')

        # --- AJUSTE DE EJES X (Limpio de negativos) ---
        for ax in ax_row:
            # Forzamos que el eje X empiece en 0
            ax.set_xlim(left=0) 
            
            # Ticks: Solo valores positivos y relevantes
            current_ticks = [t for t in ax.get_xticks() if t >= 0]
            # Añadimos los límites calculados y limpiamos duplicados
            custom_ticks = sorted(list(set(current_ticks + [l_inf, l_sup])))
            ax.set_xticks(custom_ticks)
            ax.tick_params(axis='x', rotation=45)

        # Títulos
        for j, suffix in enumerate(["Dispersión", "Distribución"]):
            ax_row[j].set_title(f"{title} | {suffix}", weight="bold")

    plt.suptitle(f"Análisis Diagnóstico: '{feature}'", fontsize=22, weight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(output_path / f"analisis_{feature}")
    plt.close()

In [ ]:
feature_analysis(df, DatasetKeys.CONSUMO_RATIO, output_path=global_path)

In [ ]:
feature_analysis(df_barrios, DatasetKeys.CONSUMO_RATIO, output_path=barrios_path)

## Outliers

### Alto Consumo (cualq. contrato)

In [ ]:
def get_outliers_report(df, feature, labels_col):
    """
    Identifica y devuelve las filas que superan el límite superior de IQR.
    """
    df = df.copy()
    report_list = []
    for label in global_labels:
        subset = df[df[labels_col] == label]
        q1, q3 = subset[feature].quantile([0.25, 0.75])
        l_sup = q3 + 1.5 * (q3 - q1)
        
        outliers_subset = subset[subset[feature] > l_sup].copy()
        outliers_subset["Threshold"] = l_sup
        report_list.append(outliers_subset)

    full_report = pd.concat(report_list).drop_duplicates(subset=df.columns)
    return full_report.sort_values(by=feature, ascending=False)

Podemos ver que mayoritariamente, los valores con ***alto ratio de consumo*** corresponden con contratos de tipo **NO DOMESTICO**. El motivo se debe a que barrios como **LA ALCORAYA** constan de zonas de regadío. Podríamos tener en cuenta que este suceso a la hora de diseñar el sistema de créditos **Water2Creds**. 

Sorprende observar que $\frac{23}{57} \approx 40.35 \%$ de nuestros barrios contienen ratios inusuales en contratos **DOMESTICOS**.

In [ ]:
outliers_df_barrios = get_outliers_report(df_barrios, DatasetKeys.CONSUMO_RATIO, DatasetKeys.USO)

print(f"Se han detectado {len(outliers_df_barrios)} filas anómalas.")
outliers_df_barrios

In [ ]:
outliers_df = get_outliers_report(df, DatasetKeys.CONSUMO_RATIO, DatasetKeys.USO)

print(f"Se han detectado {len(outliers_df)} filas anómalas.")
outliers_df.head(10)

### Consumo (contrato DOMESTICO)

Es interesante analizar que cuando estamos tratando antes contratos de uso **DOMESTICO**, el consumo se eleva en aquellos meses de calor. En cambio, al observar contratos de tipo **NO DOMESTICO** el consumo elevado puede incluir tanto meses fríos como cálidos.

In [ ]:
outliers_domestico = outliers_df[outliers_df[DatasetKeys.USO] == "DOMESTICO"] \
                                     .sort_values(DatasetKeys.CONSUMO_RATIO, ascending=False) \
                                     .groupby(DatasetKeys.BARRIO) \
                                     .head(1)

outliers_domestico.sort_values(by=DatasetKeys.CONSUMO_RATIO, ascending=False)

In [ ]:
outliers_domestico = outliers_df_barrios[outliers_df_barrios[DatasetKeys.USO] == "DOMESTICO"] \
                                     .sort_values(DatasetKeys.CONSUMO_RATIO, ascending=False) \
                                     .groupby(DatasetKeys.BARRIO) \
                                     .head(1)

outliers_domestico.sort_values(by=DatasetKeys.CONSUMO_RATIO, ascending=False)

## Estadística Descriptiva

In [ ]:
def plot_univariate_distributions(df, features,output_path):
    n_features = len(features)
    cols = min(n_features, 2)
    rows = (n_features + cols - 1) // cols 

    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows), squeeze=False)
    axes = axes.flatten()

    colors = ["skyblue", "lightgreen", "salmon", "plum"]

    for idx in range(len(axes)):
        if idx < n_features:
            feature = features[idx]
            ax = axes[idx]
            
            sns.histplot(data=df, x=feature, kde=True, bins=20, ax=ax, color=colors[idx % 4])
            ax.set_title(f"Distribución de {feature}")
            ax.set_xlabel("Valor")
            ax.set_ylabel("Frecuencia")
        else:
            fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.savefig(output_path / f"distribuciones_columnas", dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
plot_univariate_distributions(df, features=[DatasetKeys.CONSUMO_RATIO], output_path=global_path)

In [ ]:
plot_univariate_distributions(df_barrios, features=[DatasetKeys.CONSUMO_RATIO], output_path=barrios_path)

## Scatter Plots

In [ ]:
def plot_usage_analysis(df, palette, output_path=None):
    sns.set_theme(style="ticks")
    fig = plt.figure(figsize=(12, 16))
    gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.2)
    lims = {"x": (0, df[DatasetKeys.NUM_CONTRATOS].max() * 1.05), 
            "y": (0, df[DatasetKeys.CONSUMO].max() * 1.05)}


    for i, label in enumerate(global_labels):
        data = df[df[DatasetKeys.USO] == label]
        current_color = palette[label]

        ratio_local = (data[DatasetKeys.CONSUMO] / data[DatasetKeys.NUM_CONTRATOS]).mean()

        for j in range(2):
            ax = fig.add_subplot(gs[i, j])
            
            sns.scatterplot(data=data, x=DatasetKeys.NUM_CONTRATOS, y=DatasetKeys.CONSUMO, 
                            ax=ax, s=30, alpha=0.7, color=current_color)
            
        
            x_max = data[DatasetKeys.NUM_CONTRATOS].max() if j == 0 else lims["x"][1]
            x_vals = np.array([0, x_max])
            y_vals = ratio_local * x_vals
            ax.plot(x_vals, y_vals, color="red", linestyle="--", linewidth=1.5, label=f"Ratio: {ratio_local:.2f}")

            if j == 1: 
                ax.set(xlim=lims["x"], ylim=lims["y"], ylabel="")        
            ax.set_title(f"{label} ({"Global" if j==1 else "Local"})", fontsize=10, weight="bold")

    ax_full = fig.add_subplot(gs[3, :])
    sns.scatterplot(data=df, x=DatasetKeys.NUM_CONTRATOS, y=DatasetKeys.CONSUMO, 
                    hue=DatasetKeys.USO, hue_order=global_labels,  palette=global_palette, ax=ax_full, s=40, alpha=0.6)

    ax_full.set_title("Comparativa Inter-Categoría", fontsize=12, weight="bold")
    sns.despine()
    plt.savefig(output_path / "scatter_plots.png")
    plt.close()

In [ ]:
plot_usage_analysis(df, global_palette, global_path)

In [ ]:
plot_usage_analysis(df_barrios, barrios_palette, barrios_path)

## Imbalance y Representatividad

In [ ]:
label = DatasetKeys.USO
df[label].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.countplot(data=df, x=label, palette="viridis", hue=label, ax=ax,edgecolor="black",linewidth=1)
ax.set_title("Distribución de Contratos por label de Uso", fontsize=14, weight="bold", pad=15)
ax.set_xlabel("label de Uso", fontsize=12)
ax.set_ylabel("Cantidad de Contratos", fontsize=12)

sns.despine()
plt.savefig(images_path / "balance_clases.png", dpi=300, bbox_inches="tight")
plt.close()